## Task 3: Build a Modular, Logged ETL System with Full Enrichment and Idempotency

This task involves creating a robust ETL pipeline structured into four reusable functions: `extract()`, `clean()`, `transform()`, and `load()`. The system will include error handling, logging, data enrichment, summary generation, and idempotent loading to CSV and MySQL.

In [11]:
# Import necessary libraries
import requests
import pandas as pd
import logging
import os

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

### 1. Extract Function

The `extract` function will be responsible for fetching raw data from a public API. It will include error handling for network issues, HTTP errors, and JSON parsing. Logging will be used to track the function's actions and data volume.

In [12]:
def extract(url: str, timeout: int = 10) -> pd.DataFrame:
    """
    Extracts data from a given URL using requests.

    Args:
        url (str): The URL of the API endpoint.
        timeout (int): Timeout for the request in seconds.

    Returns:
        pd.DataFrame: A DataFrame containing the extracted data, or an empty DataFrame if extraction fails.
    """
    logger.info(f"Starting data extraction from: {url}")
    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
        data = response.json()
        df = pd.DataFrame(data)
        logger.info(f"Successfully extracted {len(df)} rows.")
        return df
    except requests.exceptions.Timeout:
        logger.error(f"Request to {url} timed out after {timeout} seconds.")
    except requests.exceptions.HTTPError as e:
        logger.error(f"HTTP error occurred during extraction from {url}: {e}")
    except requests.exceptions.ConnectionError as e:
        logger.error(f"Connection error occurred during extraction from {url}: {e}")
    except requests.exceptions.RequestException as e:
        logger.error(f"An unknown error occurred during extraction from {url}: {e}")
    except ValueError as e:
        logger.error(f"Failed to decode JSON from response for {url}: {e}")
    return pd.DataFrame() # Return empty DataFrame on failure

# Example usage of the extract function
# Update the API_URL to the TVMaze API
API_URL = 'https://api.tvmaze.com/shows'
raw_df = extract(API_URL)

if not raw_df.empty:
    print("Extracted Data Head from TVMaze API:")
    display(raw_df.head())
else:
    print("No data extracted from TVMaze API.")

2026-05-15 19:11:48,898 - INFO - Starting data extraction from: https://api.tvmaze.com/shows


2026-05-15 19:11:50,582 - INFO - Successfully extracted 240 rows.


Extracted Data Head from TVMaze API:


,id,url,name,type,language,genres,status,runtime,averageRuntime,premiered,...,rating,weight,network,webChannel,dvdCountry,externals,image,summary,updated,_links
0,1,https://www.tvmaze.com/shows/1/under-the-dome,Under the Dome,Scripted,English,"[Drama, Science-Fiction, Thriller]",Ended,60.0,60,2013-06-24,...,{'average': 6.6},100,"{'id': 2, 'name': 'CBS', 'country': {'name': '...",None,None,"{'tvrage': 25988, 'thetvdb': 264492, 'imdb': '...",{'medium': 'https://static.tvmaze.com/uploads/...,<p><b>Under the Dome</b> is the story of a sma...,1769177765,{'self': {'href': 'https://api.tvmaze.com/show...
1,2,https://www.tvmaze.com/shows/2/person-of-interest,Person of Interest,Scripted,English,"[Action, Crime, Science-Fiction]",Ended,60.0,60,2011-09-22,...,{'average': 8.8},100,"{'id': 2, 'name': 'CBS', 'country': {'name': '...",None,None,"{'tvrage': 28376, 'thetvdb': 248742, 'imdb': '...",{'medium': 'https://static.tvmaze.com/uploads/...,<p>You are being watched. The government has a...,1778595370,{'self': {'href': 'https://api.tvmaze.com/show...
2,3,https://www.tvmaze.com/shows/3/bitten,Bitten,Scripted,English,"[Drama, Horror, Romance]",Ended,60.0,60,2014-01-11,...,{'average': 7.4},97,"{'id': 7, 'name': 'CTV Sci-Fi Channel', 'count...",None,None,"{'tvrage': 34965, 'thetvdb': 269550, 'imdb': '...",{'medium': 'https://static.tvmaze.com/uploads/...,<p>Based on the critically acclaimed series of...,1704793709,{'self': {'href': 'https://api.tvmaze.com/show...
3,4,https://www.tvmaze.com/shows/4/arrow,Arrow,Scripted,English,"[Drama, Action, Science-Fiction]",Ended,60.0,60,2012-10-10,...,{'average': 7.4},99,"{'id': 5, 'name': 'The CW', 'country': {'name'...",None,None,"{'tvrage': 30715, 'thetvdb': 257655, 'imdb': '...",{'medium': 'https://static.tvmaze.com/uploads/...,"<p>After a violent shipwreck, billionaire play...",1766868900,{'self': {'href': 'https://api.tvmaze.com/show...
4,5,https://www.tvmaze.com/shows/5/true-detective,True Detective,Scripted,English,"[Drama, Crime, Thriller]",Running,60.0,63,2014-01-12,...,{'average': 8.1},100,"{'id': 8, 'name': 'HBO', 'country': {'name': '...",None,None,"{'tvrage': 31369, 'thetvdb': 270633, 'imdb': '...",{'medium': 'https://static.tvmaze.com/uploads/...,<p>Touch darkness and darkness touches you bac...,1777231753,{'self': {'href': 'https://api.tvmaze.com/show...


### 2. Clean Function

The `clean` function will take the raw DataFrame and perform necessary data cleaning operations. This includes normalizing nested JSON structures, handling missing values, and ensuring appropriate data types. Logging will track the number of rows before and after cleaning.

In [14]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans the raw DataFrame by normalizing nested JSON fields, handling missing values, and converting types.

    Args:
        df (pd.DataFrame): The raw DataFrame from the extract step.

    Returns:
        pd.DataFrame: The cleaned DataFrame.
    """
    logger.info(f"Starting data cleaning. Received {len(df)} rows.")

    if df.empty:
        logger.warning("Input DataFrame for cleaning is empty.")
        return pd.DataFrame()

    cleaned_df = df.copy()

    # Normalize nested 'rating' column
    if 'rating' in cleaned_df.columns:
        cleaned_df['rating_average'] = cleaned_df['rating'].apply(lambda x: x.get('average') if isinstance(x, dict) else None)
        cleaned_df = cleaned_df.drop(columns=['rating'])

    # Normalize 'schedule' column
    if 'schedule' in cleaned_df.columns:
        cleaned_df['schedule_time'] = cleaned_df['schedule'].apply(lambda x: x.get('time') if isinstance(x, dict) else None)
        cleaned_df['schedule_days'] = cleaned_df['schedule'].apply(lambda x: ', '.join(x.get('days')) if isinstance(x, dict) and isinstance(x.get('days'), list) else None)
        cleaned_df = cleaned_df.drop(columns=['schedule'])

    # Columns to drop directly as requested or if they are original nested structures
    columns_to_drop = ['network', 'image', '_links', 'externals', 'dvdCountry']
    for col in columns_to_drop:
        if col in cleaned_df.columns:
            cleaned_df = cleaned_df.drop(columns=[col])

    # Convert 'premiered' and 'ended' to datetime, coerce errors
    for col in ['premiered', 'ended']:
        if col in cleaned_df.columns:
            cleaned_df[col] = pd.to_datetime(cleaned_df[col], errors='coerce')

    # Fill missing numerical values with 0 (example for rating_average)
    if 'rating_average' in cleaned_df.columns:
        cleaned_df['rating_average'] = cleaned_df['rating_average'].fillna(0)

    # Fill missing categorical values with 'Unknown'
    # Note: 'network_name', 'network_country', 'image_medium' are no longer created, so removed from fillna list.
    for col in ['genres', 'language', 'type', 'status', 'schedule_days']:
        if col in cleaned_df.columns:
            if cleaned_df[col].dtype == 'object':
                cleaned_df[col] = cleaned_df[col].fillna('Unknown')

    # Remove HTML tags from 'summary' column
    if 'summary' in cleaned_df.columns:
        cleaned_df['summary'] = cleaned_df['summary'].str.replace(r'<.*?>', '', regex=True)
        cleaned_df['summary'] = cleaned_df['summary'].fillna('')

    logger.info(f"Finished data cleaning. Outputted {len(cleaned_df)} rows.")
    return cleaned_df

# Example usage of the clean function
cleaned_df = clean(raw_df)

if not cleaned_df.empty:
    print("\nCleaned Data Head:")
    display(cleaned_df.head())
    print("\nCleaned Data Info:")
    cleaned_df.info()
else:
    print("No data after cleaning.")

2026-05-15 19:12:00,038 - INFO - Starting data cleaning. Received 240 rows.
2026-05-15 19:12:00,048 - INFO - Finished data cleaning. Outputted 240 rows.



Cleaned Data Head:


,id,url,name,type,language,genres,status,runtime,averageRuntime,premiered,ended,officialSite,weight,webChannel,summary,updated,rating_average,schedule_time,schedule_days
0,1,https://www.tvmaze.com/shows/1/under-the-dome,Under the Dome,Scripted,English,"[Drama, Science-Fiction, Thriller]",Ended,60.0,60,2013-06-24,2015-09-10,http://www.cbs.com/shows/under-the-dome/,100,None,Under the Dome is the story of a small town th...,1769177765,6.6,22:00,Thursday
1,2,https://www.tvmaze.com/shows/2/person-of-interest,Person of Interest,Scripted,English,"[Action, Crime, Science-Fiction]",Ended,60.0,60,2011-09-22,2016-06-21,http://www.cbs.com/shows/person_of_interest/,100,None,You are being watched. The government has a se...,1778595370,8.8,22:00,Tuesday
2,3,https://www.tvmaze.com/shows/3/bitten,Bitten,Scripted,English,"[Drama, Horror, Romance]",Ended,60.0,60,2014-01-11,2016-04-15,http://bitten.space.ca/,97,None,Based on the critically acclaimed series of no...,1704793709,7.4,22:00,Friday
3,4,https://www.tvmaze.com/shows/4/arrow,Arrow,Scripted,English,"[Drama, Action, Science-Fiction]",Ended,60.0,60,2012-10-10,2020-01-28,NaN,99,None,"After a violent shipwreck, billionaire playboy...",1766868900,7.4,21:00,Tuesday
4,5,https://www.tvmaze.com/shows/5/true-detective,True Detective,Scripted,English,"[Drama, Crime, Thriller]",Running,60.0,63,2014-01-12,NaT,https://www.max.com/shows/true-detective/9a4a3...,100,None,Touch darkness and darkness touches you back. ...,1777231753,8.1,21:00,Sunday



Cleaned Data Info:
<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id              240 non-null    int64         
 1   url             240 non-null    str           
 2   name            240 non-null    str           
 3   type            240 non-null    str           
 4   language        240 non-null    str           
 5   genres          240 non-null    object        
 6   status          240 non-null    str           
 7   runtime         229 non-null    float64       
 8   averageRuntime  240 non-null    int64         
 9   premiered       240 non-null    datetime64[us]
 10  ended           218 non-null    datetime64[us]
 11  officialSite    162 non-null    str           
 12  weight          240 non-null    int64         
 13  webChannel      14 non-null     object        
 14  summary         240 non-null    str           
 1

### 3. Transform Function

The `transform` function will take the cleaned DataFrame, engineer new features (calculated columns), and generate a summary table using `groupby()`. Logging will track the operations and the number of rows.

In [15]:
def transform(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transforms the cleaned DataFrame by creating new features and generating a summary table.

    Args:
        df (pd.DataFrame): The cleaned DataFrame.

    Returns:
        pd.DataFrame: The transformed DataFrame with new calculated columns.
    """
    logger.info(f"Starting data transformation. Received {len(df)} rows.")

    if df.empty:
        logger.warning("Input DataFrame for transformation is empty.")
        return pd.DataFrame()

    transformed_df = df.copy()

    # --- Engineer at least 3 new calculated columns ---

    # 1. 'premier_year': Year when the show premiered
    if 'premiered' in transformed_df.columns:
        transformed_df['premier_year'] = transformed_df['premiered'].dt.year.fillna(0).astype(int)
        logger.info("Added 'premier_year' column.")

    # 2. 'runtime_category': Categorize shows based on runtime
    if 'runtime' in transformed_df.columns:
        def categorize_runtime(runtime):
            if pd.isna(runtime):
                return 'Unknown'
            elif runtime <= 30:
                return 'Short'
            elif runtime <= 60:
                return 'Medium'
            else:
                return 'Long'
        transformed_df['runtime_category'] = transformed_df['runtime'].apply(categorize_runtime)
        logger.info("Added 'runtime_category' column.")

    # 3. 'summary_word_count': Word count of the summary
    if 'summary' in transformed_df.columns:
        transformed_df['summary_word_count'] = transformed_df['summary'].apply(lambda x: len(str(x).split()) if pd.notna(x) else 0)
        logger.info("Added 'summary_word_count' column.")

    # 4. 'show_duration_years': Calculate the duration of the show in years (if 'ended' is available)
    if 'premiered' in transformed_df.columns and 'ended' in transformed_df.columns:
        # Convert 'ended' to datetime, handling NaT from previous cleaning if any
        transformed_df['ended_dt'] = pd.to_datetime(transformed_df['ended'], errors='coerce')
        transformed_df['show_duration_years'] = (transformed_df['ended_dt'].dt.year - transformed_df['premiered'].dt.year)
        # For shows that are still running or ended in the same year, set duration to at least 1
        transformed_df['show_duration_years'] = transformed_df['show_duration_years'].apply(lambda x: max(x, 1) if pd.notna(x) else 0)
        transformed_df = transformed_df.drop(columns=['ended_dt'])
        logger.info("Added 'show_duration_years' column.")

    # --- Generate a printed summary table using groupby() ---

    logger.info("Generating summary table.")
    summary_table = None
    # Changed grouping column from 'network_name' to 'language'
    if 'language' in transformed_df.columns and 'rating_average' in transformed_df.columns:
        summary_table = transformed_df.groupby('language').agg(
            mean_rating=('rating_average', 'mean'),
            min_rating=('rating_average', 'min'),
            max_rating=('rating_average', 'max'),
            total_shows=('id', 'count')
        ).reset_index()
        logger.info("Summary table generated by 'language'.")

    logger.info(f"Finished data transformation. Outputted {len(transformed_df)} rows.")

    # Return transformed DataFrame and optionally the summary table
    return transformed_df, summary_table

# Example usage of the transform function
transformed_df, show_summary = transform(cleaned_df)

if not transformed_df.empty:
    print("\nTransformed Data Head (with new columns):")
    display(transformed_df.head())
    print("\nTransformed Data Info:")
    transformed_df.info()

if show_summary is not None and not show_summary.empty:
    print("\nSummary Table by Language:")
    display(show_summary.head())
else:
    print("No data after transformation or summary table could not be generated.")



2026-05-15 19:12:06,358 - INFO - Starting data transformation. Received 240 rows.
2026-05-15 19:12:06,364 - INFO - Added 'premier_year' column.
2026-05-15 19:12:06,367 - INFO - Added 'runtime_category' column.
2026-05-15 19:12:06,372 - INFO - Added 'summary_word_count' column.
2026-05-15 19:12:06,384 - INFO - Added 'show_duration_years' column.
2026-05-15 19:12:06,384 - INFO - Generating summary table.
2026-05-15 19:12:06,405 - INFO - Summary table generated by 'language'.
2026-05-15 19:12:06,406 - INFO - Finished data transformation. Outputted 240 rows.



Transformed Data Head (with new columns):


,id,url,name,type,language,genres,status,runtime,averageRuntime,premiered,...,webChannel,summary,updated,rating_average,schedule_time,schedule_days,premier_year,runtime_category,summary_word_count,show_duration_years
0,1,https://www.tvmaze.com/shows/1/under-the-dome,Under the Dome,Scripted,English,"[Drama, Science-Fiction, Thriller]",Ended,60.0,60,2013-06-24,...,None,Under the Dome is the story of a small town th...,1769177765,6.6,22:00,Thursday,2013,Medium,57,2.0
1,2,https://www.tvmaze.com/shows/2/person-of-interest,Person of Interest,Scripted,English,"[Action, Crime, Science-Fiction]",Ended,60.0,60,2011-09-22,...,None,You are being watched. The government has a se...,1778595370,8.8,22:00,Tuesday,2011,Medium,96,5.0
2,3,https://www.tvmaze.com/shows/3/bitten,Bitten,Scripted,English,"[Drama, Horror, Romance]",Ended,60.0,60,2014-01-11,...,None,Based on the critically acclaimed series of no...,1704793709,7.4,22:00,Friday,2014,Medium,75,2.0
3,4,https://www.tvmaze.com/shows/4/arrow,Arrow,Scripted,English,"[Drama, Action, Science-Fiction]",Ended,60.0,60,2012-10-10,...,None,"After a violent shipwreck, billionaire playboy...",1766868900,7.4,21:00,Tuesday,2012,Medium,85,8.0
4,5,https://www.tvmaze.com/shows/5/true-detective,True Detective,Scripted,English,"[Drama, Crime, Thriller]",Running,60.0,63,2014-01-12,...,None,Touch darkness and darkness touches you back. ...,1777231753,8.1,21:00,Sunday,2014,Medium,47,0.0



Transformed Data Info:
<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   id                   240 non-null    int64         
 1   url                  240 non-null    str           
 2   name                 240 non-null    str           
 3   type                 240 non-null    str           
 4   language             240 non-null    str           
 5   genres               240 non-null    object        
 6   status               240 non-null    str           
 7   runtime              229 non-null    float64       
 8   averageRuntime       240 non-null    int64         
 9   premiered            240 non-null    datetime64[us]
 10  ended                218 non-null    datetime64[us]
 11  officialSite         162 non-null    str           
 12  weight               240 non-null    int64         
 13  webChannel           1

,language,mean_rating,min_rating,max_rating,total_shows
0,English,7.452966,0.0,9.2,236
1,Japanese,7.850000,7.3,8.8,4


### 4. Load Function

The `load` function will save the transformed data to a CSV file and a MySQL database. It will implement idempotency for the database load to prevent duplicate rows upon re-execution. Logging will track the loading process.

In [16]:

import mysql.connector
from mysql.connector import Error
import os

# Database configuration (replace with your MySQL details)
# It's recommended to store these securely, e.g., in environment variables
MYSQL_HOST = 'localhost'
MYSQL_DATABASE = 'etl_db'
MYSQL_USER = 'root'
MYSQL_PASSWORD = os.getenv("DB_PASSWORD")
MYSQL_TABLE = 'tvmaze_shows'


In [24]:
def load(df: pd.DataFrame, csv_filepath: str, mysql_table_name: str):
    """
    Loads the transformed DataFrame to a CSV file and a MySQL database.
    Ensures idempotency for MySQL by checking for existing primary keys.

    Args:
        df (pd.DataFrame): The transformed DataFrame.
        csv_filepath (str): Path to save the CSV file.
        mysql_table_name (str): Name of the table in the MySQL database.
    """
    logger.info(f"Starting data loading. Received {len(df)} rows.")

    if df.empty:
        logger.warning("Input DataFrame for loading is empty.")
        return

def load(df: pd.DataFrame, csv_filepath: str, mysql_table_name: str):
    """
    Loads the transformed DataFrame to a CSV file and a MySQL database.
    Ensures idempotency for MySQL by checking for existing primary keys.

    Args:
        df (pd.DataFrame): The transformed DataFrame.
        csv_filepath (str): Path to save the CSV file.
        mysql_table_name (str): Name of the table in the MySQL database.
    """
    logger.info(f"Starting data loading. Received {len(df)} rows.")

    if df.empty:
        logger.warning("Input DataFrame for loading is empty.")
        return

    # --- Load to CSV ---
    try:
        df.to_csv(csv_filepath, index=False)
        logger.info(f"Data successfully loaded to CSV: {csv_filepath}")
    except Exception as e:
        logger.error(f"Error loading data to CSV: {e}")

    # --- Load to MySQL with Idempotency ---
    conn = None
    cursor = None
    try:
        conn = mysql.connector.connect(
            host=MYSQL_HOST,
            database=MYSQL_DATABASE,
            user=MYSQL_USER,
            password=MYSQL_PASSWORD
        )

        if conn.is_connected():
            cursor = conn.cursor()

            # Create database if it doesn't exist
            cursor.execute(f"CREATE DATABASE IF NOT EXISTS {MYSQL_DATABASE}")
            conn.database = MYSQL_DATABASE

            # Dynamically create table schema based on DataFrame columns and dtypes
            # Assuming 'id' is the primary key and will be used for idempotency
            columns_sql = []
            for col, dtype in df.dtypes.items():
                if col == 'id':
                    columns_sql.append(f"`{col}` INT PRIMARY KEY")
                elif 'datetime' in str(dtype):
                    columns_sql.append(f"`{col}` DATETIME")
                elif 'float' in str(dtype):
                    columns_sql.append(f"`{col}` FLOAT")
                elif 'int' in str(dtype):
                    columns_sql.append(f"`{col}` INT")
                elif col == 'summary':  # ← ADD THIS
                    columns_sql.append(f"`{col}` TEXT CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci")
                else: # Default to VARCHAR for objects
                    max_len = df[col].dropna().astype(str).apply(len).max() if not df[col].empty else 255
                    columns_sql.append(f"`{col}` VARCHAR({min(max_len * 2, 1000)}) CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci")
            create_table_query = f"CREATE TABLE IF NOT EXISTS `{mysql_table_name}` ({', '.join(columns_sql)})"
            cursor.execute(create_table_query)
            logger.info(f"Table '{mysql_table_name}' checked/created in MySQL.")

            # Prepare for insertion
            cols = df.columns.tolist()
            placeholders = ', '.join(['%s'] * len(cols))
            insert_query = f"INSERT INTO `{mysql_table_name}` ({', '.join([f'`{c}`' for c in cols])}) VALUES ({placeholders})"

            # Idempotency check: only insert if 'id' does not already exist
            # For larger datasets, consider INSERT ... ON DUPLICATE KEY UPDATE
            # or a batching approach with pre-checking existing IDs.
            rows_inserted = 0
            for index, row in df.iterrows():
                # Check if the primary key (id) already exists
                check_query = f"SELECT COUNT(*) FROM `{mysql_table_name}` WHERE `id` = %s"
                cursor.execute(check_query, (row['id'],))
                if cursor.fetchone()[0] == 0:
                    # Convert pandas NaT to None for MySQL
                    insert_data = []
                    for val in row.values:
                        try:
                            if pd.isna(val):
                                insert_data.append(None)
                            else:
                                insert_data.append(str(val) if isinstance(val, list) else val)
                        except (TypeError, ValueError):
                            # If pd.isna() fails (e.g., on a list), just convert to string
                            insert_data.append(str(val))
                    cursor.execute(insert_query, insert_data)
                    rows_inserted += 1
                else:
                    logger.debug(f"Row with id {row['id']} already exists, skipping insertion.")

            conn.commit()
            logger.info(f"Successfully inserted {rows_inserted} new rows into MySQL table '{mysql_table_name}'.")
            logger.info(f"Total rows in DataFrame for loading: {len(df)}. Rows skipped (already exist): {len(df) - rows_inserted}.")

    except Error as e:
        logger.error(f"Error connecting to or interacting with MySQL: {e}")
    finally:
        if conn and conn.is_connected():
            if cursor:
                cursor.close()
            conn.close()
            logger.info("MySQL connection closed.")

# --- Full ETL Pipeline Execution ---

def run_etl_pipeline():
    logger.info("--- Starting ETL Pipeline ---")

    # 1. Extract
    api_url = 'https://api.tvmaze.com/shows'
    extracted_data = extract(api_url)

    if extracted_data.empty:
        logger.error("Extraction failed. Aborting pipeline.")
        return

    # 2. Clean
    cleaned_data = clean(extracted_data)

    if cleaned_data.empty:
        logger.error("Cleaning resulted in empty data. Aborting pipeline.")
        return

    # 3. Transform
    transformed_data, summary_output = transform(cleaned_data)

    if transformed_data.empty:
        logger.error("Transformation resulted in empty data. Aborting pipeline.")
        return

    # Display summary
    if summary_output is not None and not summary_output.empty:
        print("\nFinal Summary Table:")
        display(summary_output)
    else:
        print("\nNo summary table generated.")

    # 4. Load
    csv_output_path = 'tvmaze_shows.csv'
    mysql_target_table = 'tvmaze_shows'
    load(transformed_data, csv_output_path, mysql_target_table)

  





In [25]:
# Re-executing the full ETL pipeline after MySQL setup
logger.info("Attempting to re-run the ETL pipeline after MySQL setup.")
run_etl_pipeline()

2026-05-15 19:29:19,248 - INFO - Attempting to re-run the ETL pipeline after MySQL setup.
2026-05-15 19:29:19,249 - INFO - --- Starting ETL Pipeline ---
2026-05-15 19:29:19,249 - INFO - Starting data extraction from: https://api.tvmaze.com/shows
2026-05-15 19:29:20,709 - INFO - Successfully extracted 240 rows.
2026-05-15 19:29:20,711 - INFO - Starting data cleaning. Received 240 rows.
2026-05-15 19:29:20,720 - INFO - Finished data cleaning. Outputted 240 rows.
2026-05-15 19:29:20,720 - INFO - Starting data transformation. Received 240 rows.
2026-05-15 19:29:20,722 - INFO - Added 'premier_year' column.
2026-05-15 19:29:20,725 - INFO - Added 'runtime_category' column.
2026-05-15 19:29:20,727 - INFO - Added 'summary_word_count' column.
2026-05-15 19:29:20,731 - INFO - Added 'show_duration_years' column.
2026-05-15 19:29:20,732 - INFO - Generating summary table.
2026-05-15 19:29:20,745 - INFO - Summary table generated by 'language'.
2026-05-15 19:29:20,746 - INFO - Finished data transforma


Final Summary Table:


,language,mean_rating,min_rating,max_rating,total_shows
0,English,7.452966,0.0,9.2,236
1,Japanese,7.850000,7.3,8.8,4


2026-05-15 19:29:20,756 - INFO - Starting data loading. Received 240 rows.
2026-05-15 19:29:20,770 - INFO - Data successfully loaded to CSV: tvmaze_shows.csv
2026-05-15 19:29:20,830 - INFO - Table 'tvmaze_shows' checked/created in MySQL.
2026-05-15 19:29:20,860 - INFO - MySQL connection closed.


MySQLInterfaceError: Python type dict cannot be converted